# 05 · A mini eval harness

How to score model-generated code against hidden tests, each
run in its **own isolated sandbox** — the essence of a code benchmark.

**Key concepts**
- **Why sandboxes matter for evals** — model code is untrusted; running each
  solution in a disposable, isolated environment is safe and reproducible.
- **Hidden tests** — the model sees the task prompt but not the tests; the tests are
  the ground-truth reward signal (pass/fail).
- **Per-task isolation** — a fresh `Sandbox` per task means no state leaks between
  solutions.

**How it works:** for each task, Claude (Bedrock) generates a solution; the harness
runs `solution + tests` in a new sandbox and records pass/fail into a scoreboard.

**Prerequisites:** AWS credentials + Bedrock model access.

In [ ]:
%pip install -q raitbox

### Configuration & generator
Bedrock client plus `generate_solution`, which asks the model for just a function body.

In [ ]:
import os
import re

import boto3

from raitbox import Sandbox

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")
MODEL_ID = os.environ.get("ANTHROPIC_BEDROCK_MODEL", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
bedrock = boto3.client("bedrock-runtime", region_name=REGION)

SYSTEM_PROMPT = [{"text": (
    "You are an expert Python programmer. Implement exactly the requested function. "
    "Reply with ONLY the function definition — no prose, no markdown fences, no tests."
)}]


def generate_solution(prompt: str) -> str:
    resp = bedrock.converse(modelId=MODEL_ID, system=SYSTEM_PROMPT,
                            messages=[{"role": "user", "content": [{"text": prompt}]}],
                            inferenceConfig={"maxTokens": 1024, "temperature": 0})
    text = "".join(b.get("text", "") for b in resp["output"]["message"]["content"])
    fenced = re.findall(r"```(?:python)?\n(.*?)```", text, re.DOTALL)
    return (fenced[0] if fenced else text).strip()

### The tasks
Each task has a `prompt` the model sees and hidden `tests` it doesn't — the ground-truth pass/fail signal.

In [ ]:
# Each task: a prompt for the model + hidden tests it never sees.
TASKS = [
    {
        "name": "is_prime",
        "prompt": "Implement `def is_prime(n: int) -> bool` returning whether n is prime.",
        "tests": "assert [n for n in range(20) if is_prime(n)] == [2,3,5,7,11,13,17,19]",
    },
    {
        "name": "run_length_encode",
        "prompt": "Implement `def rle(s: str) -> str` run-length encoding, e.g. 'aaabb' -> 'a3b2'.",
        "tests": "assert rle('aaabb') == 'a3b2' and rle('') == '' and rle('x') == 'x1'",
    },
    {
        "name": "two_sum",
        "prompt": "Implement `def two_sum(nums, target)` returning indices [i,j] with nums[i]+nums[j]==target.",
        "tests": "r = two_sum([2,7,11,15], 9); assert sorted(r) == [0,1]",
    },
]

## Run each solution in its own isolated sandbox

In [ ]:
def evaluate(task) -> tuple[bool, str]:
    solution = generate_solution(task["prompt"])
    program = solution + "\n" + task["tests"] + "\nprint('ALL_TESTS_PASSED')"
    with Sandbox.create(region_name=REGION) as sbx:   # fresh, isolated per task
        ex = sbx.run_code(program)
    if ex.error is not None:
        return False, f"{ex.error.name}: {ex.error.value}"
    ok = "ALL_TESTS_PASSED" in "".join(ex.logs.stdout)
    return ok, "passed" if ok else "assertion failed"


results = [(t["name"], *evaluate(t)) for t in TASKS]

print("\n=== scoreboard ===")
for name, ok, detail in results:
    print(f"{'PASS' if ok else 'FAIL'}  {name:20} {detail}")
score = sum(ok for _, ok, _ in results)
print(f"\nscore: {score}/{len(results)}")

## Recap & where to go next
You built a small, isolated benchmark runner with a pass/fail scoreboard.

**Related:** `07` (pass@k over many samples) · `10` (parallel, dataset × agent) ·
`06`/`14`/`15` (harder, iterative and patch-based evals).